# TP 7 : jointures réparties

date de modification: 25/03/2026

# Introduction

On considère un SGBD réparti composé d'un SGBD local S0 et de
2 sources distantes : S1 et S2.

L'objectif de ce TP est d'étudier l'évaluation des requêtes accédant à des données situées dans plusieurs sources. Il s'agit de comprendre l'évaluation d'une requête de jointure entre 2 relations qui sont situées sur deux sources  distinctes.

*   Définir un schéma global décrivant l'ensemble des données et sur lequel un utlisateur peut formuler une requête globale.
*   Implanter l'évaluation répartie d'une requête globale
*   Comprendre l'ordre et l'emplacement des opérations permettant d'évaluer une requête répartie : savoir expliquer quel site traite quelles opérations.


# Préparation

installer (ou mettre à jour) DuckDB


In [ ]:
# duckdb est déjà installé sur les machines de colab
# !pip install duckdb --pre
#!pip install -U duckdb

installer un parser SQL

In [ ]:
pip install sqlglot

Affichage des résultats

In [ ]:
pip install -q itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.0 MB/s eta 0:00:00


In [ ]:
# pour améliorer l'affichage des tableaux contenant le resultat d'une requete
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

Les lib nécessaires pour ce TP

In [ ]:
import duckdb
# import sqlite3
import time
import os

import pandas as pd
import os
import shutil


Les dossiers servant de zone de **transfert de données** entre les sites

In [ ]:
local_dir = "tmp"
os.makedirs(os.path.join(local_dir, "S0"), exist_ok=True)
os.makedirs(os.path.join(local_dir, "S1"), exist_ok=True)
os.makedirs(os.path.join(local_dir, "S2"), exist_ok=True)
os.listdir(local_dir)

['S0', 'S1', 'S2']

Connections aux sources. Chaque source est un SGBD indépendant

In [ ]:
S0 = duckdb.connect('base0.db')
S1 = duckdb.connect('base1.db')
S2 = duckdb.connect('base2.db')

# Créer les données


## Site S1 : Garage, Personne

On crée les tables du site S1

In [ ]:
create_tables_1 = """
    drop table if exists Garage cascade;
    drop table if exists Personne cascade;

    drop sequence if exists id_garage;
    drop sequence if exists id_pers;

    CREATE SEQUENCE if not exists id_garage START 1;
    CREATE SEQUENCE if not exists id_pers START 1;

    CREATE TABLE if not exists Garage (
    idgarage INT DEFAULT nextval('id_garage'),
    nom VARCHAR(255),
    ville VARCHAR(255),
    jourdefermeture VARCHAR(255),
    PRIMARY KEY (idgarage)
    );

    CREATE TABLE if not exists Personne (
    idpers INT DEFAULT nextval('id_pers'),
    nom VARCHAR(255) NOT NULL,
    prenom VARCHAR(255),
    age INT,
    PRIMARY KEY (idpers)
    );
"""
S1.execute(create_tables_1)

In [ ]:
insertion_1 = """
INSERT INTO Garage (nom, ville, jourdefermeture) VALUES ('Speedy', 'Paris', 'Dimanche'),
('AutoService', 'Tours', 'Lundi'), ('Feu Vert', 'Paris', 'Dimanche'),
('Garage lyonnais', 'Lyon', 'Samedi'), ('Modern Garage', 'Bordeaux', 'Lundi'),
('Garage Auto', 'Paris', 'Dimanche'), ('La Centrale', 'Marseille', 'Lundi'),
('AD Garage', 'Paris', 'Dimanche'), ('Euromaster', 'Lyon', 'Samedi'),
('Nauroto', 'La Rochelle', 'Dimanche');

INSERT INTO Personne (nom, prenom, age) VALUES ('Armand', 'Rébecca', 24),
('Hebert', 'Aimée', 35), ('Ribeiro', 'Marielle', 18), ('Savary', 'Hilaire', 27),
('Dupont', 'Jean', 64), ('Dubois', 'Léo', 23), ('Mbappé', 'Kylian', 21),
('Georges', 'Antoine', 25), ('Cavani', 'Edinson', 32),
('Messi', 'Lionel', 33), ('Precieuse', 'Pierre', 45),
('Durand','Jeanne', 18), ('Zoulou','Zoé', 18),
('Pitaut','Enzo', 18), ('Hunter','Paulo', 18);
"""

S1.execute(insertion_1)

Tables de S1

In [ ]:
S1.execute("select table_name from duckdb_tables()").fetchall()

[('Garage',), ('Personne',)]

Shéma de S1

In [ ]:
# S1.execute("describe Garage").df()

In [ ]:
shema_base = """
  SELECT t.table_name, c.column_name
  FROM duckdb_columns() c
  JOIN duckdb_tables() t ON (c.table_name = t.table_name)
  ORDER BY t.table_name
"""

S1.execute(shema_base).df()

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
S1.execute("select * from Garage").df()

,idgarage,nom,ville,jourdefermeture
0,1,Speedy,Paris,Dimanche
1,2,AutoService,Tours,Lundi
2,3,Feu Vert,Paris,Dimanche
3,4,Garage lyonnais,Lyon,Samedi
4,5,Modern Garage,Bordeaux,Lundi
5,6,Garage Auto,Paris,Dimanche
6,7,La Centrale,Marseille,Lundi
7,8,AD Garage,Paris,Dimanche
8,9,Euromaster,Lyon,Samedi
9,10,Nauroto,La Rochelle,Dimanche


In [ ]:
S1.execute("select * from Personne").df()

Loading ITables v2.7.3 from the internet... (need help?)


## Site 2 : Mecanicien

In [ ]:
create_tables_2 = """
drop table if exists Mecanicien cascade;

CREATE TABLE if not exists Mecanicien (
    idpers INT,
    idgarage INT NOT NULL,
    niveau INT,
    PRIMARY KEY (idpers),
    );
"""
S2.execute(create_tables_2)

In [ ]:
insertion_2 = """
INSERT INTO Mecanicien (idpers, idgarage, niveau) VALUES
(1, 1, 2), (2, 1, 3), (4, 10, 2), (5, 10, 1), (6, 3, 2), (8, 6, 2),
(10, 3, 1), (11, 4, 2), (12, 4, 2), (13, 1, 2);
 """

S2.execute(insertion_2)

In [ ]:
S2.execute(shema_base).df()

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
S2.execute("select * from Mecanicien").df()

,idpers,idgarage,niveau
0,1,1,2
1,2,1,3
2,4,10,2
3,5,10,1
4,6,3,2
5,8,6,2
6,10,3,1
7,11,4,2
8,12,4,2
9,13,1,2


## SQL Parser

Le parseur SQL vous permet d'analyser l'expression SQL d'une requête pour obtenir facilement le nom des tables accédées par la requête, la liste des prédicats de sélection et de jointure ainsi que les attributs projetés.

Pour faciliter le TP, veillez à écrire les requêtes SQL en respectant les règles suivantes:

*    Toujour utiliser la notation pointée: alias.attribut pour mentionner un attribut, ceci dans le SELECT, le WHERE, et le ON.
*    Dans le FROM : définir un alias pour chaque table
*    Définir les jointures avec les mots JOIN ON


In [ ]:
from sqlglot import parse_one, exp
from sqlglot.expressions import Column, Select, Join, Where, Table, EQ, LT, LTE, GT, GTE, NEQ, Between, Literal, And, Or, Not
from sqlglot.optimizer.normalize import normalize

class Query:
  def __init__(self, query):
    self.table_names = list()
    self.table_aliases = {}
    self.projections = list()
    self.joins = list()
    self.selections = list()
    # self.query_ast = parse_one(query)
    self.query_ast = normalize(parse_one(query))

    # tables dans la clause FROM
    for table in self.query_ast.find_all(Table):
      self.table_names.append(table.name)
      if table.alias is not None:
        self.table_aliases[table.alias] = table.name

    # projections dans la clause SELECT
    for attr in self.query_ast.expressions:
      table = self.table_aliases.get(attr.table, attr.table)
      self.projections.append( (table, attr.name))

    # jointures dans la clause JOIN

    # # ancienne version ?
    # for j in self.query_ast.args['joins']:
    #   pred = j.args['on']
    #   left_table = self.table_aliases.get(pred.left.table,pred.left.table)
    #   right_table = self.table_aliases.get(pred.right.table,pred.right.table)
    #   self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))

    for j in self.query_ast.find_all(Join):
      join_conds = []
      join_tables = set()
      if j.on(): # jointure avec le mot clé ON (donc ce n'est pas un natural join)
        condition = j.args.get("on")
        if isinstance(condition, And):  # Si conjonction (AND) de plusieurs predicats
            conditions = condition.flatten()
        else:
            conditions = [condition]

        for expr in conditions:
            join_conds.append(expr.sql())
            # Extraire les tables des colonnes impliquées
            for col in expr.find_all(Column):
                join_tables.add(col.table)
        self.joins.append( ( join_conds, join_tables))

      # pred = j.args['on']
      # left_table = self.table_aliases.get(pred.left.table,pred.left.table)
      # right_table = self.table_aliases.get(pred.right.table,pred.right.table)
      # self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))
      # # if isinstance(pred, EQ):
      # #   print("equi jointure")
      # # else:
      # #   print("theta jointure")


    # sélections dans la clause WHERE
    # Liste des types autorisés
    COMPARISON_OPS = (EQ, LT, LTE, GT, GTE, NEQ)

    for pred in self.query_ast.find(exp.Where).find_all(exp.Predicate):
      if isinstance(pred, COMPARISON_OPS) :
        left_table = self.table_aliases.get(pred.left.table, pred.left.table)
        if isinstance(pred.right, Literal):
          self.selections.append( (pred.sql(), left_table) )
        elif isinstance(pred.right, Column):
          right_table = self.table_aliases.get(pred.right.table,pred.right.table)
          self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))
          # if isinstance(pred, EQ):
          #   print("équi-jointure")
          # else:
          #   print("théta-jointure")
        else:
          print("predicat non pris en compte dans le TP !!! ", pred )
      elif isinstance(pred, exp.Between) or isinstance(pred, exp.In):
        table = self.table_aliases.get(pred.this.table, pred.this.table)
        self.selections.append( (pred.sql(), table))
      # elif isinstance(pred, exp.In):
      #   table = self.table_aliases.get(pred.this.table, pred.this.table)
      #   pred = pred.args
      else:
        print("predicat non pris en compte dans le TP !!! ", pred )

  def get_tables(self):
      return self.table_names

  def get_aliases(self):
      return self.table_aliases

  def get_projections(self):
      return self.projections

  def get_joins(self):
      return self.joins

  def get_selections(self):
        return self.selections

  def get_ast(self):
        return self.query_ast

  def get_ast_string(self):
        return repr(self.query_ast)


print("class Query defined")


R1 = """
SELECT g.nom, g.ville
FROM Garage g
JOIN Mecanicien m ON g.idgarage = m.idgarage
WHERE m.niveau = 2
"""

# print(repr(parse_one(R1)))

query = Query(R1)
print("table names:", query.get_tables())
print("table aliases:", query.get_aliases())
print("projections:", query.get_projections())
print("joins:", query.get_joins())

class Query defined
table names: ['Garage', 'Mecanicien']
table aliases: {'g': 'Garage', 'm': 'Mecanicien'}
projections: [('Garage', 'nom'), ('Garage', 'ville')]
joins: [(['g.idgarage = m.idgarage'], {'g', 'm'})]


Exemple d'utilisation du parseur SQL pour R1

In [ ]:
R1 = """
SELECT g.nom, g.ville
FROM Garage g
JOIN Mecanicien m ON g.idgarage = m.idgarage
WHERE m.niveau = 2
"""

# R2 = """
# SELECT g.nom, g.ville
# FROM Garage g
# JOIN Mecanicien m ON g.idgarage = m.idgarage
# WHERE (m.niveau between 1 and 2) AND m.idpers <> 10
# """

# R3 = """
# SELECT g.nom, g.ville
# FROM Garage g
# JOIN Mecanicien m ON g.idgarage = m.idgarage
# WHERE m.niveau in (1,2)
# """

query = Query(R1)

# print("arbre syntaxique de la requete", query.get_ast_string())

print("table names:", query.get_tables())
print("table aliases:", query.get_aliases())
print("projections:", query.get_projections())
print("joins:", query.get_joins())
print("selections:", query.get_selections())

table names: ['Garage', 'Mecanicien']
table aliases: {'g': 'Garage', 'm': 'Mecanicien'}
projections: [('Garage', 'nom'), ('Garage', 'ville')]
joins: [(['g.idgarage = m.idgarage'], {'g', 'm'})]
selections: [('m.niveau = 2', 'Mecanicien')]


# Exercice 1

Proposer une solution pour traiter les requêtes suivantes, telle que :

*  les sous-requêtes sont traitées sur S1 ou S2 autant que possible,

*  les résultats intermédiaires peuvent être transférés dans S0 pour obtenir le résultat final.


## Question 1

La requete globale R1 est

SELECT g.nom, g.ville

FROM Garage g

JOIN Mecanicien m ON g.idgarage = m.idgarage

WHERE m.niveau = 2

Compléter le plan d'exécution réparti proposé ci-dessous,  tel que :
*   site **S2** : M = sous-requête sur la table Mecanicien,
*   transférer le résultat de M vers le site S1 contenant Garage,
*   site **S1** : R1 = sous-requête avec jointure,
*   transférer le résultat de R1 vers S0.

In [ ]:
M = S2.execute("SELECT m.idgarage FROM Mecanicien as m WHERE m.niveau = 2").df()
M_at_S2  = os.path.join(local_dir, "S2", "M")
M.to_csv(M_at_S2, index=False)

Transférer le résultat de la requête M depuis le site S2 vers le site S1

In [ ]:
print("transfert vers S1, nombre de tuples:", len(M))
M=None

M_at_S1 = os.path.join(local_dir, "S1", "M")
shutil.move(M_at_S2, M_at_S1)
# on considère que c'est équivalent à un transfert distant (par scp) depuis S2 vers S1

transfert vers S1, nombre de tuples: 7


'tmp/S1/M'

Importer le résultat dans le SGBD de S1

In [ ]:
# S1.execute("create or replace table T1 as select cast(idgarage as int) as idgarage from read_csv('/tmp/S1/T1', AUTO_DETECT=TRUE)")
S1.execute(f"create or replace table M as select * from read_csv('{M_at_S1}', AUTO_DETECT=TRUE)")
S1.execute("select * from M").df()

,idgarage
0,1
1,10
2,3
3,6
4,4
5,4
6,1


In [ ]:
# compléter
R1 = S1.execute("SELECT g.nom, g.ville FROM Garage as g JOIN M m ON g.idgarage = m.idgarage").df()
R1_at_S1 = os.path.join(local_dir, "S1", "R1")
R1.to_csv(R1_at_S1, index=False)
R1

,nom,ville
0,Speedy,Paris
1,Feu Vert,Paris
2,Garage lyonnais,Lyon
3,Garage Auto,Paris
4,Nauroto,La Rochelle
5,Speedy,Paris
6,Garage lyonnais,Lyon


transférer le résultat de S1 vers S0

In [ ]:
print("transfert vers S0, nombre de tuples:", len(R1))
R1_at_S0 = os.path.join(local_dir, "S0", "R1")
R1=None
shutil.move(R1_at_S1, R1_at_S0)

transfert vers S0, nombre de tuples: 7


'tmp/S0/R1'

In [ ]:
S0.execute(f"SELECT * from  read_csv('{R1_at_S0}', AUTO_DETECT=TRUE)").df()

,nom,ville
0,Speedy,Paris
1,Feu Vert,Paris
2,Garage lyonnais,Lyon
3,Garage Auto,Paris
4,Nauroto,La Rochelle
5,Speedy,Paris
6,Garage lyonnais,Lyon


## Question 2

Requête R2:

SELECT g.ville, g.nom, p.idpers

FROM Mecanicien m JOIN Garage g ON m.idgarage = g.idgarage JOIN Personne p ON p.idpers = m.idpers

WHERE p.age > 18

AND g.jourdefermeture = 'Dimanche'

ORDER BY g.ville, g.nom


In [ ]:
P = S1.execute("SELECT p.idpers FROM Personne as p WHERE p.age > 18").df()
P_at_S1  = os.path.join(local_dir, "S1", "P")
P.to_csv(P_at_S1, index=False)

P_at_S2 = os.path.join(local_dir, "S2", "P")
shutil.move(P_at_S1, P_at_S2)
S2.execute(f"create or replace table P as select * from read_csv('{P_at_S2}', AUTO_DETECT=TRUE)")

G = S1.execute("SELECT g.ville, g.nom, g.idgarage FROM Garage as g WHERE g.jourdefermeture = 'Dimanche'").df()
G_at_S1  = os.path.join(local_dir, "S1", "G")
G.to_csv(G_at_S1, index=False)

G_at_S2 = os.path.join(local_dir, "S2", "G")
shutil.move(G_at_S1, G_at_S2)
S2.execute(f"create or replace table G as select * from read_csv('{G_at_S2}', AUTO_DETECT=TRUE)")

# ----

R1 = S2.execute("SELECT g.ville, g.nom, p.idpers FROM Mecanicien as m JOIN P ON m.idpers = p.idpers JOIN G on m.idgarage = g.idgarage ORDER BY g.ville, g.nom").df()
R1_at_S2  = os.path.join(local_dir, "S2", "M")
R1.to_csv(M_at_S2, index=False)

R1_at_S0 = os.path.join(local_dir, "S0", "R1")
shutil.move(R1_at_S2, R1_at_S0)
S0.execute(f"SELECT * from  read_csv('{R1_at_S0}', AUTO_DETECT=TRUE)").df()

,ville,nom,idpers
0,La Rochelle,Nauroto,5
1,La Rochelle,Nauroto,4
2,Paris,Feu Vert,10
3,Paris,Feu Vert,6
4,Paris,Garage Auto,8
5,Paris,Speedy,2
6,Paris,Speedy,1


# Exercice 2

L'objectif de cet exercice est d'automatiserle traitement d'une requête de jointure entre 2 sites.


## 1) Fonction traiter_requête

Définir la fonction *traiter_requete(requete)* qui implante le plan d'exécution d'une requête répartie entre 2 tables A et B, telle que :
*  La requête peut contenir des sélections et des projections.
*  Une jointure entre A et B est définie dans une clause JOIN ON.
*  A et B ne sont pas sur le même site.
*  On commence par interroger le site **S1** pour transférer les données utiles vers **S2**.
*  penser à ne **pas** transférer les attributs inutilisés dans la requête.
*  La jointure est traitée sur le site **S2**.
*  Le résultat est transféré dans S0.


Indications, interroger les sites (cf requête *shema_base* ci-dessus) pour savoir où est stockée chaque table.


In [ ]:
import re

shema_base = """
  SELECT DISTINCT t.table_name
  FROM duckdb_columns() c
  JOIN duckdb_tables() t ON (c.table_name = t.table_name)
  ORDER BY t.table_name
"""

def remove_conditions_from_where(requete, conditions_to_remove):
    # Look for WHERE
    match = re.search(r'\bWHERE\b', requete, flags=re.IGNORECASE)
    if not match:
        return requete  # no WHERE → nothing to modify

    where_start = match.end()

    # Find where the WHERE clause ends (before ORDER BY, GROUP BY, etc.)
    tail_match = re.search(
        r'\b(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT)\b',
        requete[where_start:],
        flags=re.IGNORECASE
    )

    if tail_match:
        where_clause = requete[where_start:where_start + tail_match.start()]
        tail = requete[where_start + tail_match.start():]
    else:
        where_clause = requete[where_start:]
        tail = ""

    # Split conditions
    conditions = [c.strip() for c in re.split(r'\bAND\b', where_clause, flags=re.IGNORECASE)]

    # Remove unwanted ones
    remaining = [c for c in conditions if c and c not in conditions_to_remove]

    # Base query (before WHERE)
    base_query = requete[:match.start()].strip()

    # Rebuild
    if not remaining:
        return f"{base_query} {tail}".strip()

    new_where = ' AND '.join(remaining)
    return f"{base_query} WHERE {new_where} {tail}".strip()

def run_S1(table, query):
  alias = next((k for k, v in query.get_aliases().items() if v == table), None)

  join_attributes = list({token
                    for conditions, aliases in query.get_joins() if alias in aliases
                    for cond in conditions
                    for token in cond.replace('=', ' ').replace('<', ' ').replace('>', ' ').split()
                    if token.startswith(f"{alias}.")})
  projections_str = ', '.join(set([f"{alias}.{col}" for tbl, col in query.get_projections() if tbl == table] + join_attributes))
  selections_str = ' AND '.join(
    [cond for cond, name in query.get_selections() if name == table]
  )

  final_projection = '*' if '*' in projections_str else projections_str
  to_select = f" WHERE {selections_str}" if len(selections_str) > 0 else ""
  final_requete = f"SELECT {final_projection} FROM {table} as {alias}{to_select}"
  print("Requete S1 : " + final_requete)

  R1 = S1.execute(final_requete).df()
  R1_at_S1  = os.path.join(local_dir, "S1", "R1")
  R1.to_csv(R1_at_S1, index=False)

  R1_at_S2 = os.path.join(local_dir, "S2", "R1")
  shutil.move(R1_at_S1, R1_at_S2)
  S2.execute(f"create or replace table R1 as select * from read_csv('{R1_at_S2}', AUTO_DETECT=TRUE)")

def run_S2(table, requete, query, toReplace):
  requete_finale = requete.replace(toReplace, "R1")
  selections_to_drop = ' AND '.join(
    [cond for cond, name in query.get_selections() if name == toReplace]
  )

  # Cette ligne enlève les sélections sur la table de S1 (car elle n'est pas disponible sur S2, on a qu'un résultat partiel)
  marequete = remove_conditions_from_where(requete_finale, selections_to_drop)
  print("Requete S2 : " + marequete)

  R2 = S2.execute(marequete).df()
  R2_at_S2  = os.path.join(local_dir, "S2", "R2")
  R2.to_csv(R2_at_S2, index=False)

  R2_at_S0 = os.path.join(local_dir, "S0", "R2")
  shutil.move(R2_at_S2, R2_at_S0)
  S0.execute(f"create or replace table R2 as select * from read_csv('{R2_at_S0}', AUTO_DETECT=TRUE)")

def traiter_requete(requete):
  requete_str = requete
  requete = Query(requete)

  # Get table lists
  tables_S1 = S1.execute(shema_base).df()['table_name'].values
  tables_S2 = S2.execute(shema_base).df()['table_name'].values

  table_first = requete.get_tables()[0] if requete.get_tables()[0] in tables_S1 else requete.get_tables()[1]
  table_second = requete.get_tables()[0] if requete.get_tables()[0] in tables_S2 else requete.get_tables()[1]

  run_S1(table_first, requete)
  run_S2(table_second, requete_str, requete, table_first)

  res = S0.execute(f"SELECT * from  R2").df()
  print(res)

traiter_requete("""
SELECT g.ville, g.nom, m.idgarage
FROM Mecanicien m JOIN Garage g ON m.idgarage = g.idgarage
WHERE m.niveau = 2
AND g.jourdefermeture = 'Dimanche'
ORDER BY g.ville, g.nom
""")

Requete S1 : SELECT g.nom, g.ville, g.idgarage FROM Garage as g WHERE g.jourdefermeture = 'Dimanche'
Requete S2 : SELECT g.ville, g.nom, m.idgarage
FROM Mecanicien m JOIN R1 g ON m.idgarage = g.idgarage WHERE m.niveau = 2 ORDER BY g.ville, g.nom
         ville          nom  idgarage
0  La Rochelle      Nauroto        10
1        Paris     Feu Vert         3
2        Paris  Garage Auto         6
3        Paris       Speedy         1
4        Paris       Speedy         1



Appliquer la fonction sur la requête

In [ ]:
# ...

## 2) Transfert minimal

Définir la fonction *traiter_requête_opt* qui améliore la fonction ci-dessus afin de **minimiser les transferts**.

Choisir de traiter la jointure sur le site S1 ou S2 de telle sorte que cela minimise les transferts.
Envisager plusieurs solutions avec des transferts de S1 vers S2 ou de S2 vers S1 et traiter la solution avec le moins de transferts.


In [ ]:
def traiter_requete_opt(requete):
  requete_str = requete
  requete = Query(requete)

  tables_S1 = S1.execute(shema_base).df()['table_name'].values
  tables_S2 = S2.execute(shema_base).df()['table_name'].values

  table_first = requete.get_tables()[0] if requete.get_tables()[0] in tables_S1 else requete.get_tables()[1]
  table_second = requete.get_tables()[0] if requete.get_tables()[0] in tables_S2 else requete.get_tables()[1]

  # S1
  alias = next((k for k, v in requete.get_aliases().items() if v == table_first), None)
  join_attributes = list({token
                    for conditions, aliases in requete.get_joins() if alias in aliases
                    for cond in conditions
                    for token in cond.replace('=', ' ').replace('<', ' ').replace('>', ' ').split()
                    if token.startswith(f"{alias}.")})
  projections_str = ', '.join(set([f"{alias}.{col}" for tbl, col in requete.get_projections() if tbl == table_first] + join_attributes))
  selections_str = ' AND '.join(
    [cond for cond, name in requete.get_selections() if name == table_first]
  )
  final_projection = '*' if '*' in projections_str else projections_str
  to_select = f" WHERE {selections_str}" if len(selections_str) > 0 else ""
  final_requete = f"SELECT {final_projection} FROM {table_first} as {alias}{to_select}"

  R1 = S1.execute(final_requete).df()

  # S2
  alias2 = next((k for k, v in requete.get_aliases().items() if v == table_second), None)
  join_attributes2 = list({token
                    for conditions, aliases in requete.get_joins() if alias2 in aliases
                    for cond in conditions
                    for token in cond.replace('=', ' ').replace('<', ' ').replace('>', ' ').split()
                    if token.startswith(f"{alias2}.")})
  projections_str2 = ', '.join(set([f"{alias2}.{col}" for tbl, col in requete.get_projections() if tbl == table_second] + join_attributes2))
  selections_str2 = ' AND '.join(
    [cond for cond, name in requete.get_selections() if name == table_second]
  )
  final_projection2 = '*' if '*' in projections_str2 else projections_str2
  to_select2 = f" WHERE {selections_str2}" if len(selections_str2) > 0 else ""
  final_requete2 = f"SELECT {final_projection2} FROM {table_second} as {alias2}{to_select2}"

  R2 = S2.execute(final_requete2).df()

  print("La requete sur S1 contient " + str(len(R1)) + f" tuples : {final_requete}")
  print(R1)
  print("La requete sur S2 contient " + str(len(R2)) + f" tuples : {final_requete2}")
  print(R2)

  # Jointures
  if (len(R1) <= len(R2)):
      R1_at_S1  = os.path.join(local_dir, "S1", "R1")
      R1.to_csv(R1_at_S1, index=False)

      R1_at_S2 = os.path.join(local_dir, "S2", "R1")
      shutil.move(R1_at_S1, R1_at_S2)
      S2.execute(f"create or replace table R1 as select * from read_csv('{R1_at_S2}', AUTO_DETECT=TRUE)")

      requete_finale = requete_str.replace(table_first, "R1")
      selections_to_drop = ' AND '.join(
        [cond for cond, name in requete.get_selections() if name == table_first]
      )

      # Cette ligne enlève les sélections sur la table de S1 (car elle n'est pas disponible sur S2, on a qu'un résultat partiel)
      marequete = remove_conditions_from_where(requete_finale, selections_to_drop)
      print("Requete S2 avec jointure : " + marequete)

      R2 = S2.execute(marequete).df()
      R2_at_S2  = os.path.join(local_dir, "S2", "R2")
      R2.to_csv(R2_at_S2, index=False)

      R2_at_S0 = os.path.join(local_dir, "S0", "R2")
      shutil.move(R2_at_S2, R2_at_S0)
      S0.execute(f"create or replace table R2 as select * from read_csv('{R2_at_S0}', AUTO_DETECT=TRUE)")
      res = S0.execute(f"SELECT * from  R2").df()
  else:
      R2_at_S2  = os.path.join(local_dir, "S2", "R2")
      R2.to_csv(R2_at_S2, index=False)

      R2_at_S1 = os.path.join(local_dir, "S1", "R2")
      shutil.move(R2_at_S2, R2_at_S1)
      S1.execute(f"create or replace table R2 as select * from read_csv('{R2_at_S1}', AUTO_DETECT=TRUE)")

      requete_finale = requete_str.replace(table_second, "R2")
      selections_to_drop = ' AND '.join(
        [cond for cond, name in requete.get_selections() if name == table_second]
      )

      # Cette ligne enlève les sélections sur la table de S1 (car elle n'est pas disponible sur S2, on a qu'un résultat partiel)
      marequete = remove_conditions_from_where(requete_finale, selections_to_drop)
      print("Requete S1 avec jointure : " + marequete)

      R1 = S1.execute(marequete).df()
      R1_at_S1  = os.path.join(local_dir, "S1", "R1")
      R1.to_csv(R1_at_S1, index=False)

      R1_at_S0 = os.path.join(local_dir, "S0", "R1")
      shutil.move(R1_at_S1, R1_at_S0)
      S0.execute(f"create or replace table R1 as select * from read_csv('{R1_at_S0}', AUTO_DETECT=TRUE)")
      res = S0.execute(f"SELECT * from  R1").df()

  print(res)

traiter_requete_opt("""
SELECT g.ville, g.nom, m.idgarage
FROM Mecanicien m JOIN Garage g ON m.idgarage = g.idgarage
WHERE m.niveau = 2
AND g.jourdefermeture = 'Dimanche'
ORDER BY g.ville, g.nom
""")

print("--------------------------------------")

traiter_requete_opt("""
SELECT g.ville, g.nom, m.idgarage
FROM Mecanicien m JOIN Garage g ON m.idgarage = g.idgarage
WHERE m.niveau = 1
AND g.jourdefermeture = 'Dimanche'
ORDER BY g.ville, g.nom
""")

NameError: name 'Query' is not defined

# Exercice 3

On propose de répartir la base TPCH tq :
*   **S1** contient les fragments de Customer dont le pays est le Canada
*   **S2**  idem mais pour le pays France

Proposer une exécution répartie de la requête suivante  de telle sorte que la taille totale (en octets) des données transférées soit minimale.


SELECT *
FROM Customer
WHERE pays in ('CANADA, 'FRANCE')
ORDER BY c_acctbal
LIMIT 10
`
une requête de sélection,projection,jointure entre ces 2 tables

In [ ]:
# scale factor
# sf = 0.2
sf = 0.05

drop_tpch_tables = """
  DROP TABLE IF EXISTS customer;
  DROP TABLE IF EXISTS lineitem;
  DROP TABLE IF EXISTS nation;
  DROP TABLE IF EXISTS orders;
  DROP TABLE IF EXISTS part;
  DROP TABLE IF EXISTS partsupp;
  DROP TABLE IF EXISTS region;
  DROP TABLE IF EXISTS supplier;
  """
S1.execute(drop_tpch_tables)

print("generate database with scale factor sf = ", sf)
S1.execute(f"CALL dbgen(sf={sf})")


drop_S1 = """
  -- on garde Customer et  Nation

  DROP TABLE IF EXISTS lineitem;
  DROP TABLE IF EXISTS orders;
  DROP TABLE IF EXISTS part;
  DROP TABLE IF EXISTS partsupp;
  DROP TABLE IF EXISTS region;
  DROP TABLE IF EXISTS supplier;
  """
S1.execute(drop_S1)


requete_globale = """
               select *
               from Customer
               join Nation on c_nationkey = n_nationkey
               where n_name in ('CANADA', 'FRANCE')
               """

# fragment Customer_Canada
requete_sur_S1 = """
               create or replace table Customer_Canada as
               select *
               from Customer
               join Nation on c_nationkey = n_nationkey
               where n_name = 'CANADA'
               order by c_acctbal
               """

S1.execute(requete_sur_S1)

drop_S1 = """
  DROP TABLE IF EXISTS Customer;
  DROP TABLE IF EXISTS Nation;
  """
S1.execute(drop_S1)

print("done")


generate database with scale factor sf =  0.05
done


In [ ]:
S1.execute("SUMMARIZE Customer_Canada").df()

Loading ITables v2.7.3 from the internet... (need help?)
